# Mean, Median, And Mode

There are three common measures in statistics that are widely used to make sense of data. You will finde them as parts of more complex formulas, but they are also useful on their own. They are the mean, median, and mode. It is important to understand what they are and for what they are used (or not used).

## Setup

Before you can start you maybe need to install some packages if not already done.

In [ ]:
%pip install -r ../requirements.txt

## Mean

The mean is the average of a set of numbers. It is calculated by adding all the numbers together and then dividing by the count of numbers. 

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i = \frac{x_1+x_2+\cdots+x_n}{n}$$

Where:

- $\bar{x}$ is the mean
- $n$ is the number of observations
- $x_i$ represents each individual observation

Let's take the example with the CPU usage data:

In [ ]:
import psutil
import time

cpu_usage = []

for _ in range(10):
    cpu_usage.append(psutil.cpu_percent())
    time.sleep(1)
    
print(cpu_usage)

The mean would be calculated as follows:

In [ ]:
print(sum(cpu_usage) / len(cpu_usage))

The useful lib `numpy` has a function for that. It is always a nice little helper so maybe you should use it as often as possible.

In [ ]:
import numpy as np

print(np.mean(cpu_usage))

The mean is sensitive to outliers, which can skew the result. Only one maximum can have a significant impact on the mean if the dataset is small.

In [ ]:
cpu_usage += [100]
print(np.mean(cpu_usage))

Let's create some random but realistic data about salaries in a big city. I use the lognormal distribution for normal people like you and me and pareto distribution for the super rich. The distribution is plotted. This could be the distribution in your hometown.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = 1_000_000

# Normal distribution for the main 90% of the population
main = np.random.lognormal(mean=10, sigma=0.35, size=int(n*0.9))
main = main + 450  # Minimum salary of 450€

# Top 10%: Pareto
top = (np.random.pareto(3.0, int(n*0.1)) + 1) * 60_000

salaries = np.concatenate([main, top])

plt.hist(salaries, bins=500)
plt.xlabel('Salary (€)')
plt.ylabel('Population')
plt.xlim(0, np.percentile(salaries, 99))  # z.B. 99%-Quantil
plt.show()

What if some fictive rich guy named, let's say, Noel Muks wants to live in your city?

In [ ]:
print(f'Mean without Noel: {np.mean(salaries):_.2f} €')
salaries = np.concatenate([salaries, [14_390_000_000]]) # Adding an extreme outlier

plt.hist(salaries, bins=500)
plt.xlabel('Salary (€)')
plt.ylabel('Population')
plt.show()

m = np.mean(salaries)
print(f'Mean with Noel: {m:_.2f} €')

Ups. He breaks the plot. Everyone is now on the "poor" side of the distribution and the mean is now much higher than before.

## Median

The median is the middle value of a sorted list of numbers. If there is an even number of observations, the median is the average of the two middle numbers. 

$$
\tilde{x} =
\begin{cases}
x_{\left(\frac{n+1}{2}\right)}, & \text{wenn } n \text{ ungerade ist} \\
\frac{x_{\left(\frac{n}{2}\right)} + x_{\left(\frac{n}{2}+1\right)}}{2}, & \text{wenn } n \text{ gerade ist}
\end{cases}
$$

Where:
- $\tilde{x}$ is the median
- $n$ is the number of observations
- $x_{(i)}$ represents the $i$-th smallest observation in the sorted list

I am not a math pro, so here is some explanation:

1. First, sort the data.
   The notation $x_{(i)}$ means the $i$-th smallest value in the sorted list.
2. If $n$ is odd:
   There is exactly one middle value, at position $\frac{n+1}{2}$.
3. If $n$ is even:
   There are two middle values, at positions $\frac{n}{2}$ and $\frac{n}{2}+1$.
   The median is the average of those two values.

**Quick examples**

1. Odd case: ([1, 3, 7, 9, 10]), ($n=5$)
   Median position: ($\frac{5+1}{2}=3$)
   Median: ($x_{(3)}=7$)

1. Even case: ([1, 3, 7, 9]), ($n=4$)
   Middle positions: ($\frac{4}{2}=2$) and ($\frac{4}{2}+1=3$)
   Median: ($\frac{x_{(2)}+x_{(3)}}{2}=\frac{3+7}{2}=5$)

The intuition is that the median splits sorted data into a lower half and an upper half, so it is more robust to outliers than the mean.

In [ ]:
salaries.sort()
n_current = len(salaries)

# Important!: Math indexing starts at 1, but Python indexing starts at 0. We do not need to add 1 to the index.
if n_current % 2 == 0:
    median = (salaries[n_current // 2 - 1] + salaries[n_current // 2]) / 2
else:
    median = salaries[n_current // 2]

print(f'Median: {median:_.2f} €')

# There is also a numpy function for the median:
print(f'Median: {np.median(salaries):_.2f} €')

You can see that the median is not affected by the extreme outlier. You can also see that the median is much lower than the mean, which is a sign of a skewed distribution. The mean is pulled up by the extreme outlier, while the median remains more representative of the typical salary in the city.

## Mode

The mode is the value that appears most frequently in a dataset. A dataset can have one mode (unimodal), more than one mode (multimodal), or no mode at all if all values are unique. The mode is useful for categorical data where we want to know which category is the most common.

In [ ]:
def mode(values):
    counts = {}

    for v in values:
        counts[v] = counts.get(v, 0) + 1

    max_count = max(counts.values())
    modes = [v for v, c in counts.items() if c == max_count]
    return modes, max_count

values = [True, False, True, True, False]
modes, count = mode(values)
print(modes, count)   # [True] 3

# You can also use numpy for the mode:
unique_vals, counts = np.unique(values, return_counts=True)
max_count = counts.max()
modes = unique_vals[counts == max_count]

print(modes, max_count)   # [ True ] 3